### Week 1: Agentic RAG

In [27]:
from dotenv import load_dotenv
load_dotenv()
 
from openai import OpenAI
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from rag_helper import RAGBase
import json

In [28]:
openai_client = OpenAI()

Preparation

In [29]:

print('Fetching lesson pages...')
reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name='llm-zoomcamp',
    commit_id='8c1834d',
    allowed_extensions={'md'},
    filename_filter=lambda path: '/lessons/' in path,
)
files = reader.read()
documents = [f.parse() for f in files]

Fetching lesson pages...


Question 1

In [30]:
# Q1 
print(f'Q1: {len(documents)} lesson pages')

Q1: 72 lesson pages


Question 2

In [31]:

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [32]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=3
)

search_results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

Question 3

In [33]:

assistant = RAGBase(index=index, llm_client=openai_client)
answer, usage = assistant.rag(question)
print(f'Q3: {usage.input_tokens} input tokens')

Q3: 7111 input tokens


Question 4

In [ ]:

chunks = chunk_documents(documents, size=2000, step=1000)
print(f'Q4: {len(chunks)} chunks')

Q4: 295 chunks


Question 5

In [ ]:

chunk_index = Index(text_fields=['content'], keyword_fields=['filename'])
chunk_index.fit(chunks)
chunk_assistant = RAGBase(index=chunk_index, llm_client=openai_client)
answer2, usage2 = chunk_assistant.rag(question)
ratio = usage.input_tokens / usage2.input_tokens
print(f'Q5: {usage2.input_tokens} tokens (chunked) vs {usage.input_tokens} (full) = {ratio:.1f}x fewer')

Q5: 2294 tokens (chunked) vs 7111 (full) = 3.1x fewer


Question 6

In [ ]:


def search(query: str) -> list[dict]:
    """
    Search the course content using the chunked document index.

    Args:
        query: The user's question or search query.

    Returns:
        A list of matching document chunks. Each result contains metadata such as
        filename and content from the most relevant chunks.
    """
    results = chunk_index.search(
        query,
        num_results=5
    )

    return results

In [37]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the course lessons for content matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course lessons."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [38]:
AGENT_INSTRUCTIONS = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

AGENT_Q='How does the agentic loop work, and how is it different from plain RAG?'

In [39]:
messages = [
    {'role': 'developer', 'content': AGENT_INSTRUCTIONS},
    {'role': 'user', 'content': AGENT_Q},
]

In [40]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [41]:
count = 0
while True:
    has_calls = False
    response = openai_client.responses.create(
        model='gpt-5.4-mini', input=messages, tools=[search_tool]
    )
    messages.extend(response.output)
    for item in response.output:
        if item.type == 'function_call':
            messages.append(make_call(item))
            has_calls = True
            count += 1
    if not has_calls:
        break
 
print(f'Q6: search called {count} times')

Q6: search called 4 times
